In [17]:
import numpy as np
import openmdao.api as om
import os

os.environ['OPENMDAO_REPORTS'] = 'none'


class aerodynamicsDis(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('B', val=0.)

        # Coupling parameter
        self.add_input('phi', val=0.)

        # Coupling output
        self.add_output('L', val=0.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        q = 1 # N/cm2
        C = 10 # cm
        psi = 0.05 # rad
        r = 0.9425
        theta0 = 0.26 # rad

        B = inputs['B']
        phi = inputs['phi']

        outputs['L'] = q*B*C * ((2*np.pi*(phi+psi)) + r*(1-np.cos(np.pi/2*(phi+psi)/theta0)))

class structuresDis(om.ExplicitComponent):
    def setup(self):
        # Coupling parameter
        self.add_input('L', val=0.)

        # Coupling output
        self.add_output('phi', val=0.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        C = 10 # cm
        p = 0.1111
        k1 = 4000 # N/cm
        k2 = 2000 # N/cm
        z1 = 0.2
        z2 = 0.7

        L = inputs['L']

        outputs['phi'] = ( L/(k1*(1+p)) - (L*p)/(k2*(1+p)) ) * ( 1/(C*(z2-z1)) )
        
class aerostructuresGroup(om.Group):
    def setup(self):
        cycle = self.add_subsystem('cycle', om.Group(), promotes=['*'])
        cycle.add_subsystem('aero', aerodynamicsDis(), promotes_inputs=['B', 'phi'],
                            promotes_outputs=['L'])
        cycle.add_subsystem('strux', structuresDis(), promotes_inputs=['L'],
                            promotes_outputs=['phi'])

        cycle.linear_solver = om.DirectSolver(rhs_checking=False)
        nlbgs = cycle.nonlinear_solver = om.NonlinearBlockGS()
        nlbgs.options['maxiter'] = 1000
        nlbgs.options['iprint'] = 2
        

In [22]:
prob = om.Problem()
prob.model = aerostructuresGroup()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

prob.model.add_design_var('B', lower = 0., upper = 500.)

prob.model.approx_totals()

prob.setup()

prob.set_val('B', 100.)

prob.run_model()

print('coupling vars')
print('L', prob.get_val('L'))
print('phi', prob.get_val('phi'))

print('iter count', 
      prob.model.cycle.aero.iter_count + prob.model.cycle.strux.iter_count)


=====
cycle
=====
NL: NLBGS 1 ; 356.834849 1
NL: NLBGS 2 ; 102.173344 0.286332302
NL: NLBGS 3 ; 30.1803847 0.0845780193
NL: NLBGS 4 ; 8.99252749 0.0252008107
NL: NLBGS 5 ; 2.68623291 0.00752794442
NL: NLBGS 6 ; 0.803034127 0.00225043638
NL: NLBGS 7 ; 0.24011671 0.000672907117
NL: NLBGS 8 ; 0.0718025831 0.00020122077
NL: NLBGS 9 ; 0.0214717042 6.0172666e-05
NL: NLBGS 10 ; 0.0064208952 1.7994025e-05
NL: NLBGS 11 ; 0.001920107 5.38094026e-06
NL: NLBGS 12 ; 0.000574189856 1.60911934e-06
NL: NLBGS 13 ; 0.000171706079 4.81192012e-07
NL: NLBGS 14 ; 5.1347091e-05 1.43895954e-07
NL: NLBGS 15 ; 1.53548658e-05 4.3030735e-08
NL: NLBGS 16 ; 4.59172867e-06 1.28679379e-08
NL: NLBGS 17 ; 1.37311338e-06 3.8480361e-09
NL: NLBGS 18 ; 4.10616792e-07 1.15071943e-09
NL: NLBGS 19 ; 1.22791107e-07 3.4411187e-10
NL: NLBGS 20 ; 3.67193707e-08 1.02902984e-10
NL: NLBGS 21 ; 1.09806706e-08 3.07724165e-11
NL: NLBGS Converged
coupling vars
L [502.01292281]
phi [0.01757113]
iter count 42
